# 03 — Phase 2: GQA + PagedAttention (vLLM)

**框架**: vLLM — 社区支持度最好的 PagedAttention 实现

### 为什么不自己写 PagedAttention？
原生 PyTorch 很难实现真正的分页机制。vLLM 提供：
- **自定义 CUDA kernel**：直接在分块内存上计算 attention
- **按需分配 block**：消除预分配导致的内存碎片
- **统一内存管理**：block table 跟踪物理→虚拟映射

### 预期效果
- TTFT / TPOT 与 baseline 相当（计算量相同）
- **内存碎片率大幅降低**
- **显存利用率提升** → 同样 10-12 GB 能处理更长序列

### 安装 vLLM (Jetson)
```bash
# 见 scripts/setup_vllm_jetson.sh
pip install vllm  # 或从源码编译
```

⚠️ 如果 vLLM 未安装，此 notebook 会用 HF 原生 + 内存碎片对比演示。

In [ ]:
import sys, gc, time, torch
sys.path.insert(0, '..')

from src.vllm_runner import (
    check_vllm_available, create_vllm_engine,
    run_vllm_benchmark, get_vllm_cache_stats,
)
from src.metrics import (
    measure_generation, run_benchmark, find_oom_threshold,
    print_memory_budget, JETSON_USABLE_GB,
)
from src.paged_cache import PagedKVCache
from src.dataset_utils import load_prompts

USE_VLLM = check_vllm_available()
print(f"vLLM available: {USE_VLLM}")
if not USE_VLLM:
    print("\n⚠️  vLLM 未安装。将使用 Python 参考实现演示分页概念。")
    print("    真正的 PagedAttention 性能提升需要 vLLM CUDA kernels。")
    print("    安装: 见 scripts/setup_vllm_jetson.sh")

In [ ]:
prompts = load_prompts('../results/pubmedqa_prompts.json')
print(f"Loaded {len(prompts)} prompts")

---
## Path A: vLLM Engine (真正的 PagedAttention)

In [ ]:
if USE_VLLM:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    
    # gpu_memory_utilization 要保守：
    # Jetson 16GB 统一内存，系统占 4-6GB → 可用 ~10-12GB
    # vLLM 的 0.85 是基于 "GPU专用显存"，在统一内存上要更低
    engine = create_vllm_engine(
        model_name=MODEL_NAME,
        gpu_memory_utilization=0.65,   # 保守! ~10.4 GB of 16 GB
        max_model_len=4096,
        block_size=16,
    )
    
    # PagedAttention block 统计
    cache_stats = get_vllm_cache_stats(engine)
    print(f"PagedAttention stats:")
    for k, v in cache_stats.items():
        print(f"  {k}: {v}")
else:
    print("Skipping vLLM engine creation (not installed)")

In [ ]:
if USE_VLLM:
    results_paged = run_vllm_benchmark(
        engine, prompts,
        max_new_tokens=256,
        warmup_runs=2,
    )
    
    import pandas as pd
    df_vllm = pd.DataFrame(results_paged)
    df_vllm.to_csv('../results/gqa_paged_vllm.csv', index=False)
    
    cols = ['ttft_ms', 'tpot_ms', 'total_time_ms', 'num_input_tokens', 'num_output_tokens']
    print(df_vllm[cols].describe().round(1))
    
    # 清理 vLLM engine
    del engine
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Skipping vLLM benchmark")

---
## Path B: Python 参考实现 (PagedKVCache)

如果 vLLM 不可用，用 Python 实现演示分页概念。
注意：这**不会**带来真正的性能收益，因为缺少自定义 CUDA kernel。
它仅用于测量内存碎片的差异。

In [ ]:

if not USE_VLLM:
    from src.jetson_utils import load_model_safe, aggressive_cleanup
    
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    FALLBACK_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
    DEVICE = "cuda"
    
    model, tokenizer = load_model_safe(
        MODEL_NAME,
        fallback_name=FALLBACK_NAME,
        device=DEVICE,
    )
    budget = print_memory_budget(model, DEVICE)


In [ ]:
if not USE_VLLM:
    BLOCK_SIZE = 16
    
    # OOM 探测：PagedAttention 应该比 baseline 能撑更长序列
    print("PagedKVCache OOM 探测...")
    oom_paged = find_oom_threshold(
        model, tokenizer,
        context_lengths=[256, 512, 1024, 2048, 4096, 8192, 16384],
        max_new_tokens=32,
        cache_factory=lambda: PagedKVCache(block_size=BLOCK_SIZE),
    )
    print(f"最大安全长度: {oom_paged['max_safe_length']}")
    print(f"OOM 发生在  : {oom_paged['oom_length']}")
    for r in oom_paged['results']:
        if r['status'] == 'ok':
            print(f"  ctx={r['context_length']:>6}  peak={r['peak_memory_mb']:>7.0f} MB")

In [ ]:
if not USE_VLLM:
    # Benchmark （碎片率对比是关键）
    results_paged = run_benchmark(
        model, tokenizer, prompts,
        cache_factory=lambda: PagedKVCache(block_size=BLOCK_SIZE),
        max_new_tokens=256,
        warmup_runs=2, num_runs=3,
    )
    
    import pandas as pd
    df = pd.DataFrame(results_paged)
    df['config'] = 'GQA+PagedAttn(ref)'
    df.to_csv('../results/gqa_paged.csv', index=False)
    
    print(df[['ttft_ms', 'tpot_ms', 'peak_memory_mb',
              'memory_fragmentation', 'memory_utilization']].describe().round(2))
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

---
### 关键观察

PagedAttention 的核心收益不是加速单条请求，而是：
1. **消除碎片** → 同样内存能存更多 KV Cache
2. **按需分配** → 不需要为最大长度预留内存
3. **并发友好** → 多请求共享物理 block